# Phase 1 — Convert, Verify, Diagnose, Build Splits
**Project:** *The Artist Confound: What Do Folk-Genre Classifiers Actually Learn?*

This notebook does the one-time heavy lifting. Run cells **in order, top to bottom**.

| Cell | What it does | Time |
|---|---|---|
| 1 | Config (env-var paths), RAM check | 1 min |
| 2 | Convert 15 pickles (`\|S32`, 205 GB) → float16 `.npy` (~12.8 GB) — **resumable** | **1.5–4 h** (disk I/O bound) |
| 3 | Build cleaned master metadata table, verify counts vs report | 2 min |
| 4 | Visual + numeric sanity check on the converted data | 1 min |
| 5 | **Overlap diagnostic** → find hop `k` between consecutive segments | 2 min |
| 6 | **MI(artist;genre)/H(genre)** + find the cross-style artists | 1 min |
| 7 | Build & save artist-disjoint and song-disjoint 5-fold splits | 1 min |

Cell 2 is safe to interrupt — it skips styles already converted, so if interrupted just re-run from Cell 1.


Paths come from `IKS_DATA_ROOT` / `IKS_PROC_ROOT` (defaults `./data`, `./processed`, relative to the repo root). The scripts in `scripts/phase1_*.py` are the reference implementation.

In [ ]:
# ================= CELL 1: Config + Environment check =================
import os, gc, glob, json, time, pickle, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=DeprecationWarning)

if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')   # resolve paths from the repo root
DATA_ROOT   = os.environ.get('IKS_DATA_ROOT', './data')        # holds dataset/<Style>/*.pickle
DATASET_DIR = f"{DATA_ROOT}/dataset"
PROC_ROOT   = os.environ.get('IKS_PROC_ROOT', './processed')   # converted data (~12.8 GB)
os.makedirs(PROC_ROOT, exist_ok=True)

STYLES = ["Bauls","Bhatiali","Bhavageethe","Bihu","Garba","Gidha","Kajri","Lavani",
          "Maand","Naatupura Paatu","Sohar","Sufi","Tamang Selo","Uttarakhandi","Veeragase"]

# Expected segment counts from report.md — used for verification
EXPECTED = {"Bauls":20223,"Bhatiali":24282,"Bhavageethe":27487,"Bihu":28018,"Garba":31177,
            "Gidha":18735,"Kajri":28976,"Lavani":24724,"Maand":33789,"Naatupura Paatu":7788,
            "Sohar":30821,"Sufi":38895,"Tamang Selo":37298,"Uttarakhandi":17872,"Veeragase":15458}

META_KEYS = ['genre','state','artist','gender','song','source',
             'no_of_artists','genre_id','state_id','artist_id','gender_id']

N_MELS, N_FRAMES, SR, HOP = 128, 130, 22050, 512   # librosa defaults assumed; Cell 5 verifies hop empirically

def find_pickle(style):
    c = glob.glob(os.path.join(DATASET_DIR, style, "**", "*.pickle"), recursive=True)
    assert len(c) == 1, f"{style}: expected exactly 1 pickle, found {c}"
    return c[0]

def to_str_list(a):
    return [x.decode('utf-8', 'replace') if isinstance(x, (bytes, np.bytes_)) else str(x) for x in a]

# --- environment check ---
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
disk_free_gb = shutil.disk_usage(PROC_ROOT).free / 1e9
print(f"RAM: {ram_gb:.0f} GB   |   disk free: {disk_free_gb:.0f} GB")
if ram_gb < 40:
    print("⚠️  WARNING: <40 GB RAM. The largest pickle (Sufi, 20.7 GB) may crash this runtime.")
for s in STYLES:
    find_pickle(s)
print("All 15 pickles located ✅")

In [ ]:
# ================= CELL 2: One-time conversion  |S32 (205 GB) -> float16 (~12.8 GB) =================
# Strategy: load one pickle fully (you have high RAM), decode strings -> float32 -> float16 in
# chunks into a memmap, save per-segment metadata to parquet (written last), free RAM.
# RESUMABLE: styles whose outputs already exist (with the right row count) are skipped.

CHUNK = 4000  # rows per conversion chunk (~2.1 GB of |S32 per chunk slice)

def already_done(style, n_expected=None):
    xp = f"{PROC_ROOT}/X_{style}.npy"
    mp = f"{PROC_ROOT}/meta_{style}.parquet"
    if not (os.path.exists(xp) and os.path.exists(mp)):
        return False
    try:
        n_x = np.load(xp, mmap_mode='r').shape[0]
        n_m = len(pd.read_parquet(mp))
        ok = (n_x == n_m) and (n_expected is None or n_x == n_expected)
        return ok
    except Exception:
        return False

t_all = time.time()
for style in STYLES:
    if already_done(style, EXPECTED.get(style)):
        print(f"[skip] {style:<16} already converted")
        continue

    t0 = time.time()
    src = find_pickle(style)
    print(f"[load] {style:<16} reading {os.path.getsize(src)/1e9:5.1f} GB ...", flush=True)
    with open(src, 'rb') as f:
        d = pickle.load(f)                      # slow: disk I/O dominates
    mel = d['mel_spec']                         # (N,128,130) dtype |S32 (ASCII strings)
    N = mel.shape[0]
    assert mel.shape[1:] == (N_MELS, N_FRAMES), f"unexpected shape {mel.shape}"

    # ---- metadata (decode bytes -> str) ----
    meta = pd.DataFrame({k: to_str_list(d[k]) for k in META_KEYS})
    meta.insert(0, 'style', style)
    meta.insert(1, 'row', np.arange(N, dtype=np.int64))   # row within this style's X file

    # ---- spectrograms: string -> float32 -> float16, chunked, into a memmap ----
    local_x = f"{PROC_ROOT}/X_{style}.npy"
    X = np.lib.format.open_memmap(local_x, mode='w+', dtype=np.float16, shape=(N, N_MELS, N_FRAMES))
    for i in range(0, N, CHUNK):
        X[i:i+CHUNK] = mel[i:i+CHUNK].astype(np.float32).astype(np.float16)
    X.flush(); del X

    # free the big pickle (keeps peak RAM low)
    del d, mel; gc.collect()

    # ---- quick verification on a random sample ----
    Xv = np.load(local_x, mmap_mode='r')
    idx = np.random.default_rng(0).choice(N, size=min(64, N), replace=False)
    samp = Xv[np.sort(idx)].astype(np.float32)
    assert np.isfinite(samp).all(), "NaN/Inf after conversion!"
    assert samp.min() >= -80.5 and samp.max() <= 0.5, f"range off: [{samp.min()},{samp.max()}]"
    del Xv

    # ---- persist metadata last (marks the style as complete) ----
    meta.to_parquet(f"{PROC_ROOT}/meta_{style}.parquet")

    exp = EXPECTED.get(style)
    flag = "✅" if (exp is None or exp == N) else f"⚠️ expected {exp}"
    print(f"[done] {style:<16} N={N:>6} {flag}   {time.time()-t0:6.0f}s   "
          f"float16 size={os.path.getsize(local_x)/1e9:.2f} GB", flush=True)

print(f"\nAll conversions finished in {(time.time()-t_all)/60:.1f} min")
print(f"Total converted size: "
      f"{sum(os.path.getsize(f'{PROC_ROOT}/X_{s}.npy') for s in STYLES)/1e9:.2f} GB")

In [ ]:
# ================= CELL 3: Master metadata table (clean + verify) =================
meta = pd.concat([pd.read_parquet(f"{PROC_ROOT}/meta_{s}.parquet") for s in STYLES],
                 ignore_index=True)

# ---------- cleaning (documented data-quality fixes) ----------
for c in ['genre','state','artist','gender','song','source']:
    meta[c] = meta[c].str.strip()                         # 'Bhatiali ', 'Tamil Nadu ', 'Female '
meta['gender'] = meta['gender'].replace({'Feamle': 'Female'})   # Bihu typo
meta['genre']  = meta['genre'].replace({'Giddha': 'Gidha'})     # xlsx/folder drift
for c in ['no_of_artists','genre_id','state_id','artist_id','gender_id']:
    meta[c] = meta[c].astype(int)
meta['gender_id'] = meta['gender_id'].replace({2: 0, 3: 0})     # dirty ids -> Female
meta['global_idx'] = np.arange(len(meta))

meta.to_parquet(f"{PROC_ROOT}/meta_all.parquet")

# ---------- verification ----------
print(f"TOTAL segments: {len(meta):,}   (report says 385,543)")
assert len(meta) == 385543, "segment total mismatch vs report — investigate before continuing"

rec = meta.drop_duplicates('source')
print(f"TOTAL recordings (unique source): {len(rec)}   (report says 605)")
print(f"TOTAL artists (unique artist_id): {meta.artist_id.nunique()}   (report says 126)\n")

tbl = (meta.groupby('style')
           .agg(segments=('global_idx','count'),
                recordings=('source','nunique'),
                artists=('artist_id','nunique'))
           .assign(expected=lambda t: t.index.map(EXPECTED))
           .assign(match=lambda t: np.where(t.segments == t.expected, '✅', '⚠️')))
print(tbl.to_string())
print("\nGender (segments):"); print(meta.gender.value_counts().to_string())

In [ ]:
# ================= CELL 4: Sanity — do the converted spectrograms look right? =================
rng = np.random.default_rng(1)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, s in zip(axes, rng.choice(STYLES, 3, replace=False)):
    X = np.load(f"{PROC_ROOT}/X_{s}.npy", mmap_mode='r')
    i = int(rng.integers(0, X.shape[0]))
    seg = X[i].astype(np.float32)
    ax.imshow(seg, aspect='auto', origin='lower', cmap='magma', vmin=-80, vmax=0)
    ax.set_title(f"{s} · seg {i}\nmin={seg.min():.1f} max={seg.max():.1f} mean={seg.mean():.1f}")
    ax.set_xlabel("frame"); ax.set_ylabel("mel bin")
plt.tight_layout(); plt.show()
# Expect: dB in [-80, 0], visible harmonic stacks / percussion verticals, max ≈ 0 (per-segment ref=max)

In [ ]:
# ================= CELL 5: OVERLAP DIAGNOSTIC — find the hop k =================
# For consecutive segments (i, i+1) of the SAME recording, slide segment i left by k frames and
# measure mean |difference| against segment i+1. A sharp minimum near 0 at some k* means:
#     hop_seconds = k* x 512 / 22050,   overlap = (130 - k*) / 130
# This single number decides how much of the 385k segments is redundant, and justifies the
# R1 (random split) -> R2 (song-disjoint) step in the paper.

def best_k_for_pair(a, b, kmax=65):
    errs = np.array([np.abs(a[:, k:] - b[:, :N_FRAMES-k]).mean() for k in range(1, kmax)])
    return np.arange(1, kmax), errs

results, curves = [], []
for s in ["Bauls", "Garba", "Kajri", "Sufi", "Uttarakhandi"]:
    sub = meta[meta['style'] == s]
    src = sub.source.value_counts().index[0]              # longest recording of this style
    rows = np.sort(sub[sub.source == src].row.values)
    X = np.load(f"{PROC_ROOT}/X_{s}.npy", mmap_mode='r')
    for j in [5, len(rows)//2, len(rows)-10]:             # three pairs spread across the recording
        a = X[rows[j]].astype(np.float32); b = X[rows[j+1]].astype(np.float32)
        ks, errs = best_k_for_pair(a, b)
        kstar, emin = int(ks[errs.argmin()]), float(errs.min())
        results.append({"style": s, "pair": j, "k*": kstar, "min_err_dB": round(emin, 3),
                        "median_err_dB": round(float(np.median(errs)), 2)})
        curves.append((s, ks, errs))

res = pd.DataFrame(results)
print(res.to_string(index=False))

kstar = int(res["k*"].mode()[0])
hop_s = kstar * HOP / SR
print(f"\n>>> Consensus k* = {kstar}  ->  hop ≈ {hop_s:.3f} s,  "
      f"segment overlap ≈ {(N_FRAMES - kstar)/N_FRAMES*100:.0f}%")
print(">>> Interpretation: if min_err_dB << median_err_dB (e.g. <1 dB vs ~10 dB), overlap is CONFIRMED")
print(">>>                and a random-segment split (R1) leaks near-duplicate audio into test.")

plt.figure(figsize=(8, 4))
for s, ks, errs in curves:
    plt.plot(ks, errs, alpha=0.5, label=s)
plt.axvline(kstar, ls='--', c='k'); plt.xlabel("shift k (frames)"); plt.ylabel("mean |Δ| (dB)")
plt.title(f"Alignment error vs shift — consensus k*={kstar} (hop≈{hop_s:.2f}s)")
handles, labels = plt.gca().get_legend_handles_labels()
by = dict(zip(labels, handles)); plt.legend(by.values(), by.keys(), fontsize=8)
plt.tight_layout(); plt.savefig(f"{PROC_ROOT}/fig_overlap_diagnostic.png", dpi=150); plt.show()
print(f"Figure saved -> {PROC_ROOT}/fig_overlap_diagnostic.png  (this goes in the paper)")

In [ ]:
# ================= CELL 6: THE STRUCTURAL CONFOUND — MI(artist;genre)/H(genre) =================
from sklearn.metrics import mutual_info_score
from scipy.stats import entropy

rec = meta.drop_duplicates('source').copy()      # recording level, n=605

mi = mutual_info_score(rec['artist_id'], rec['genre_id'])          # nats
H  = entropy(rec['genre_id'].value_counts(normalize=True))         # nats
print(f"I(artist; genre) = {mi:.4f} nats")
print(f"H(genre)         = {H:.4f} nats")
print(f"RATIO I/H        = {mi/H:.4f}   <-- abstract number: ≈1.0 means artist identity is a")
print(f"                                    sufficient statistic for genre (the Bayes shortcut).\n")

# ---- cross-style artists: the ONLY within-subject genre variation in the corpus ----
span = (rec.groupby('artist_id')
           .agg(artist=('artist', 'first'),
                n_genres=('genre', 'nunique'),
                genres=('genre', lambda g: sorted(set(g))),
                n_recordings=('source', 'nunique')))
cross = span[span.n_genres > 1]
print(f"Cross-style artists found: {len(cross)}  (report says 4)")
print(cross.to_string())
cross.to_csv(f"{PROC_ROOT}/cross_style_artists.csv")

# ---- per-genre artist counts (drives fold variance; goes in the paper's data table) ----
pg = (rec.groupby('genre')
         .agg(recordings=('source', 'nunique'), artists=('artist_id', 'nunique'))
         .sort_values('artists'))
print("\nArtists per genre (small numbers => high fold variance in artist-disjoint eval):")
print(pg.to_string())

In [ ]:
# ================= CELL 7: Build & freeze the evaluation splits =================
# R1 (random segment) needs no file — it's a seeded shuffle at train time.
# Here we freeze:
#   folds_song_disjoint.json    (R2)  — genre-stratified 5-fold over recordings (grouped by source)
#   folds_artist_disjoint.json  (R3/R4) — genre-stratified round-robin over artists;
#                                cross-style artists keep ONE fold (assigned at first encounter).
SEED, K = 42, 5
rng = np.random.default_rng(SEED)
rec = meta.drop_duplicates('source').copy()

# ---------- artist-disjoint ----------
artist2fold = {}
for gid, grp in rec.groupby('genre_id'):
    artists = list(rng.permutation(sorted(grp.artist_id.unique())))
    slot = 0
    for a in artists:
        if int(a) not in artist2fold:            # cross-style artist already placed -> keep it
            artist2fold[int(a)] = slot % K
            slot += 1
rec['fold_artist'] = rec.artist_id.map(artist2fold)

# ---------- song-disjoint ----------
source2fold = {}
for gid, grp in rec.groupby('genre_id'):
    sources = list(rng.permutation(sorted(grp.source.unique())))
    for i, s in enumerate(sources):
        source2fold[s] = i % K
rec['fold_song'] = rec.source.map(source2fold)

# ---------- coverage check: does every fold's TEST set contain every genre? ----------
def coverage(col, name):
    cov = rec.pivot_table(index='genre', columns=col, values='source', aggfunc='nunique').fillna(0).astype(int)
    print(f"\n{name}: test recordings per (genre x fold)")
    print(cov.to_string())
    missing = (cov == 0).sum().sum()
    if missing:
        print(f"⚠️  {missing} (genre, fold) cells empty — with 4-artist classes (Naatupura Paatu) this is")
        print("    unavoidable at K=5. Report per-genre accuracy only over folds where the genre appears,")
        print("    and state this in the paper.")
coverage('fold_artist', 'ARTIST-DISJOINT (R3/R4)')
coverage('fold_song',  'SONG-DISJOINT (R2)')

# ---------- persist ----------
json.dump({"seed": SEED, "k": K, "artist2fold": {str(k): v for k, v in artist2fold.items()}},
          open(f"{PROC_ROOT}/folds_artist_disjoint.json", 'w'), indent=1)
json.dump({"seed": SEED, "k": K, "source2fold": source2fold},
          open(f"{PROC_ROOT}/folds_song_disjoint.json", 'w'), indent=1)

# master index every training run will load
idx = meta.merge(rec[['source', 'fold_artist', 'fold_song']], on='source', how='left')
assert idx.fold_artist.notna().all() and idx.fold_song.notna().all()
idx[['global_idx','style','row','genre','genre_id','state_id','artist_id','gender_id',
     'source','song','fold_artist','fold_song']].to_parquet(f"{PROC_ROOT}/index_with_folds.parquet")

print(f"\nSaved -> {PROC_ROOT}/folds_artist_disjoint.json")
print(f"Saved -> {PROC_ROOT}/folds_song_disjoint.json")
print(f"Saved -> {PROC_ROOT}/index_with_folds.parquet   ({len(idx):,} rows)")
print("\nPhase 1 complete ✅")

## ✅ After running everything, send me these four things
1. **Cell 5:** the consensus `k*`, the hop in seconds, the overlap %, and whether `min_err_dB << median_err_dB`.
2. **Cell 6:** the ratio **I(artist;genre)/H(genre)** (4 decimals).
3. **Cell 6:** the **cross-style artists table** (names + which two genres each spans).
4. **Cell 7:** the two **(genre × fold) coverage tables** (paste as text).

With those four I will build the Phase-2 notebook: the `Dataset`/loader over the float16 memmaps, ResNet-18 + CRNN, the four split regimes R1→R4, song-level aggregation, and the collapse-curve figure.

**Notes**
- Everything persistent lives in `$IKS_PROC_ROOT` (default `./processed`, ~13 GB). Cell 2 auto-skips converted styles.
- The original 205 GB of pickles are never modified. Once you've verified Phase 1, you *may* archive them, but keep them until Phase 2 trains successfully.
